In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "ebel2021prior")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_2 = os.path.join(original_data_pathway, "FunctFix2016_Grissini_data_extract_abeeku_data.csv")
complete_path_1 = os.path.join(original_data_pathway, "FunctFix2016_Grissini_data_glmm_data.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df1 = pd.read_csv(complete_path_1)
df1['drop_out']="false"
df2 = pd.read_csv(complete_path_2, sep=",", encoding="latin-1")

df2 = df2[df2.Subject.str.contains("Abeeku")]
df2['drop_out']="true"
# df2['Subject'].unique()

In [3]:

data_frames=[df1,df2]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    data_frames[index]=x
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)


In [4]:
fulldf.rename(columns={"subject":"participant",
                      "group":"group_original",
                      "success&attempt":"tool_use",
                      "species":"species_origina",
                      "sex":"sex_original",
                      "age":"age_original"}, inplace=True) 
fulldf['participant'] = fulldf['participant'].str.rstrip()
# fulldf.columns

In [5]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['participant'] = fulldf['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')

In [6]:
fulldf[['month','day', 'year']] = fulldf['date'].str.split('/',expand=True)

comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
fulldf= fulldf.merge(ape_dob,left_on='participant', right_on='name', how='left') 
fulldf['dodc'] = fulldf['year'].astype(str) + '-' + fulldf['month'].astype(str) + '-' + fulldf['day'].astype(str)
fulldf['dodc'] = pd.to_datetime(fulldf['dodc'])
fulldf['dob'] = pd.to_datetime(fulldf['dob'])

fulldf['age_in_years'] = (fulldf['dodc'] - fulldf['dob']).dt.days//365

In [7]:
fulldf['study_id']="ebel2021prior"
# fulldf.columns


In [8]:

studyID_standardized=fulldf[[ 'study_id','year','month', 'day', 'participant',  'age_original','age_in_years','sex', 'species',
       'session', 'trial', 'condition','drop_out', 'group_original', 'success',
       'rake_attempt', 'rake_hesitation', 'rake_toolate', 'tool_takenby',
       'mouth_beforeattempt', 'mouth_beforesuccess', 'eaten_first',
       'grissini_pref', 
       # 'tool_use', 'notes'
       ]]
comp_out_path_stand = os.path.join(out_pathway, 'ebel2021prior_standardized.csv')
studyID_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'ebel2021prior_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
